# Vitals Table EDA

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

BASE_DIR = Path('..').resolve() / 'data'
vitals = pd.read_parquet(BASE_DIR / 'vitals.parquet').copy()

for c in ['measured_at', 'created_at']:
    if c in vitals.columns:
        vitals[c] = pd.to_datetime(vitals[c], errors='coerce')

if 'measured_at' not in vitals.columns and 'created_at' in vitals.columns:
    vitals['measured_at'] = vitals['created_at']

print('shape:', vitals.shape)
vitals.head()


In [ ]:
summary = {
    'rows': len(vitals),
    'unique_residents': vitals['resident_id'].nunique() if 'resident_id' in vitals.columns else np.nan,
    'unique_facilities': vitals['facility_id'].nunique() if 'facility_id' in vitals.columns else np.nan,
    'unique_vital_types': vitals['vital_type'].nunique() if 'vital_type' in vitals.columns else np.nan,
    'measured_min': vitals['measured_at'].min() if 'measured_at' in vitals.columns else pd.NaT,
    'measured_max': vitals['measured_at'].max() if 'measured_at' in vitals.columns else pd.NaT,
}
pd.Series(summary)


In [ ]:
# Volume by month
if 'measured_at' in vitals.columns:
    d = vitals.dropna(subset=['measured_at']).copy()
    d['month'] = d['measured_at'].dt.to_period('M').dt.to_timestamp()
    s = d.groupby('month').size()

    fig, ax = plt.subplots(figsize=(10, 4))
    s.plot(ax=ax)
    ax.set_title('Vitals Events per Month')
    ax.set_xlabel('Month')
    ax.set_ylabel('Event count')
    plt.tight_layout()
    plt.show()


In [ ]:
# Top vital types
if 'vital_type' in vitals.columns:
    top = vitals['vital_type'].fillna('Unknown').value_counts().head(15).sort_values()

    fig, ax = plt.subplots(figsize=(9, 5))
    top.plot(kind='barh', ax=ax)
    ax.set_title('Top 15 Vital Types')
    ax.set_xlabel('Count')
    plt.tight_layout()
    plt.show()


In [ ]:
# Numeric value distributions
vitals['value_num'] = pd.to_numeric(vitals['value'], errors='coerce') if 'value' in vitals.columns else np.nan
if 'dystolic_value' in vitals.columns:
    vitals['dystolic_num'] = pd.to_numeric(vitals['dystolic_value'], errors='coerce')
else:
    vitals['dystolic_num'] = np.nan

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(vitals['value_num'].dropna(), bins=50, alpha=0.8)
axes[0].set_title('Value Distribution')
axes[0].set_xlabel('value')
axes[0].set_ylabel('Count')

axes[1].hist(vitals['dystolic_num'].dropna(), bins=50, alpha=0.8)
axes[1].set_title('Dystolic Value Distribution')
axes[1].set_xlabel('dystolic_value')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()


In [ ]:
# Resident monitoring intensity
if {'resident_id', 'measured_at'}.issubset(vitals.columns):
    per_res = vitals.groupby('resident_id').size()

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.hist(per_res, bins=50, alpha=0.8)
    ax.set_title('Vitals per Resident')
    ax.set_xlabel('Events per resident')
    ax.set_ylabel('Count of residents')
    plt.tight_layout()
    plt.show()

    print('Median events/resident:', per_res.median())
    print('P90 events/resident:', per_res.quantile(0.9))
